# Generation Rent · Independent QA checks

Runs every QA script in `qa/` in order and shows its real output. **Restart & Run All** reproduces the whole audit (about 5–8 minutes).

Ground rules the scripts follow:
- Originals (`datasets/`, `pipeline/`) are only **read**. The pipeline is executed on copies in `qa/sandbox/`.
- Key numbers are recomputed with a **different tool** than the pipeline used: standard-library `csv` (Phase 2, traces, Power BI checks), DuckDB SQL straight from the raw files (Phase 5), and pandas for the SQL queries (Phase 4).
- Every script also writes its results to `qa/results/*.json` / `.md`. The findings are summarised in `qa/QA_REPORT.md`.

In [1]:
import subprocess, sys, time
from pathlib import Path

QA = Path.cwd() if Path.cwd().name == "qa" else Path.cwd() / "qa"
ROOT = QA.parent

def run(script):
    """Run one QA script in an isolated interpreter (-I) from the project root and print its output."""
    t = time.time()
    p = subprocess.run([sys.executable, "-I", str(QA / script)], cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    print(p.stdout[-20000:])
    if p.returncode != 0:
        print("STDERR:\n", p.stderr[-5000:])
    print(f"--- {script}: exit code {p.returncode}, {time.time() - t:.0f}s")
    assert p.returncode == 0, f"{script} failed"

print(sys.version)
print("project root:", ROOT.name, "(folder name only; scripts use paths relative to it)")

3.13.17 (tags/v3.13.17:64e080d, Oct 21 2026, 14:08:53) [MSC v.1944 64 bit (AMD64)]
project root: DataStudy (folder name only; scripts use paths relative to it)


## 0 · Build the sandboxes and run the project's pipeline there
Copies the raw data + notebook + SQL into `qa/sandbox/`, runs the unmodified notebook on a fresh kernel, then the project's SQL runner. Also tries the notebook in its real layout (`qa/sandbox_inplace/`, no `datasets/` beside it).

In [2]:
run("p0_setup_sandbox.py")

fresh_run_in_sandbox {'exit_code': 0, 'stderr_tail': ' does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.\n  self._get_loop()\n[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.\n[NbConvertApp] Writing 153221 bytes to executed_fresh.ipynb\n'}
run_sql_all_files {'exit_code': 0, 'stderr_tail': ''}
run_in_original_layout {'exit_code': 0, 'first_error_lines': []}

--- p0_setup_sandbox.py: exit code 0, 24s


## 2 · Raw data profiling (standard library only)
Rows, columns, types, blanks, placeholders, duplicates, distributions, outliers, categorical values, dates, encoding. Full detail: `results/p2_profile.md`.

In [3]:
run("p2_profile_raw.py")

profiling hpi_a ...
profiling hpi_q ...
profiling ptir ...
profiling net_earn ...
profiling city_rents ...
profiling overburden ...
profiling age_leave ...
profiling with_parents ...
profiling hicp ...

file               rows cols ragged   bom  utf8 mojib exdup keydup  blankV nonnum comma               time fmt      min      max future gapSer
hpi_a             6,114   10      0 False  True     0     0      0     156      0     0       {'annual': 6114}     2005     2025      0      0
hpi_q            33,617   10      0 False  True     0     0      0     864      0     0   {'quarterly': 33617}  2005-Q1  2026-Q2      0      0
ptir              2,161    9      0 False  True     0     0      0       0      0     0       {'annual': 2161}     2000     2025      0      0
net_earn        199,986   11      0 False  True     0     0      0       0      0     0     {'annual': 199986}     2000     2025      0   1170
city_rents        5,416   10      0 False  True     0     0      0       0      0 

In [4]:
run("p2b_slice_checks.py")   # the exact raw slices that feed the dashboard + targeted anomalies


### house_price_index_2015
  filter: {'purchase': 'TOTAL', 'unit': 'I15_A_AVG'}
  rows_all_geos: 759
  rows_eu27_plus_avg: 567
  duplicate_geo_period: 0
  year_range: 2005-2025
  eu27_missing_entirely: ['EL']
  eu27_avg_present: True
  blank_values: 38
  flags: {'-': 556, 'e': 5, 'b': 3, 'p': 3}
  first_year_by_geo_min_max: ('2005', '2010')
  last_year_by_geo: {'2025': 27}

### net_earnings_eur
  filter: {'estruct': 'NET', 'ecase': 'P1_NCH_AW100', 'currency': 'EUR'}
  rows_all_geos: 892
  rows_eu27_plus_avg: 668
  duplicate_geo_period: 0
  year_range: 2000-2025
  eu27_missing_entirely: []
  eu27_avg_present: True
  blank_values: 0
  flags: {'-': 640, 'b': 28}
  first_year_by_geo_min_max: ('2000', '2013')
  last_year_by_geo: {'2025': 28}

### net_earnings_pps
  filter: {'estruct': 'NET', 'ecase': 'P1_NCH_AW100', 'currency': 'PPS'}
  rows_all_geos: 892
  rows_eu27_plus_avg: 668
  duplicate_geo_period: 0
  year_range: 2000-2025
  eu27_missing_entirely: []
  eu27_avg_present: True
  blank

## 3 · Notebook validation
Fresh-kernel run vs saved notebook (errors, warnings, changed outputs, hidden state, hard-coded paths, output hashes), then row-count lineage, merge validation and pandas pitfalls.

In [5]:
run("p3_notebook_run_compare.py")

1. Errors/warnings in fresh run: none
2. Code cells: 15; cells whose text output changed: 0 (+ 2 differing only in printed folder paths: cells [2, 30])
3. Saved execution counts: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] -> sequential from 1: True
4a. Names used before definition (static, by cell order): none
4b. Absolute hard-coded paths: none
4c. Path logic: BASE = cwd.parent if cwd is named 'python' else cwd; datasets/ is searched at BASE and upwards; outputs go to BASE/clean/generation-rent
5. Output CSVs identical to originals (SHA-256):
   city_rent.csv                 29933      29933 identical=True
   country_year.csv              76917      76917 identical=True
   dim_country.csv                2280       2280 identical=True
   dim_year.csv                    757        757 identical=True
   hicp_monthly.csv            7447508    7447508 identical=True
   long_age_leave.csv           114969     114969 identical=True
   long_city_rents.csv          366204     366204 i

In [6]:
run("p3_lineage_and_pitfalls.py")

raw data from: datasets
outputs to:    clean\generation-rent
hpi_a         OK  datasets\eurostat-house-price-index\prc_hpi_a.csv
hpi_q         OK  datasets\eurostat-house-price-index\prc_hpi_q.csv
ptir          OK  datasets\eurostat-house-price-to-income\tipsho60.csv
net_earn      OK  datasets\eurostat-net-earnings\earn_nt_net.csv
city_rents    OK  datasets\eurostat-city-rents\prc_colc_rents.csv
overburden    OK  datasets\eurostat-housing-cost-overburden\ilc_lvho07a.csv
age_leave     OK  datasets\eurostat-age-leaving-home\yth_demo_030.csv
with_parents  OK  datasets\eurostat-young-adults-living-with-parents\ilc_lvps08.csv
hicp          OK  datasets\eurostat-hicp-housing-food\prc_hicp_minr.csv
(2331, 10)
27 EU27 countries found (expected 27)
EU27 countries with a 2024 value: 27, of which flagged 'b': 27
(588, 25)
hicp long rows 196,331 -> wide rows 99,856
PASS one row per geo-year
PASS exactly the 27 EU countries + EU average
PASS no negative values
PASS percentages between 0 and 100
PAS

## 4 · SQL validation
Every statement executed; every result-producing query re-implemented in pandas and compared cell by cell; targeted logic checks; the original `.duckdb` opened read-only and compared with the CSVs.

In [7]:
run("p4_sql_validation.py")

query    runs    rows  pandas recomputation
00.check RUNS       8  n/a (metadata query)  
01.Q1    RUNS      25  n/a (metadata query)  
01.Q2    RUNS       5  EXACT  5 rows x 6 cols identical, same order
01.Q3    RUNS       1  EXACT  1 rows x 4 cols identical, same order
01.Q4    RUNS       6  EXACT  6 rows x 5 cols identical, same order
01.Q5    RUNS      10  EXACT  10 rows x 2 cols identical, same order
01.Q6    RUNS      44  EXACT  44 rows x 6 cols identical, same order
02.Q1    RUNS      25  EXACT  25 rows x 6 cols identical, same order
02.Q2    RUNS      28  EXACT  28 rows x 3 cols identical, same order
02.Q3    RUNS      27  EXACT  27 rows x 2 cols identical, same order
03.Q1    RUNS      26  EXACT  26 rows x 6 cols identical, same order
03.Q2    RUNS      36  EXACT  36 rows x 4 cols identical, same order
03.Q3    RUNS      13  EXACT  13 rows x 6 cols identical, same order
04.Q1    RUNS      27  EXACT  27 rows x 6 cols identical, same order
04.Q2    RUNS       3  EXACT  3 rows x 

In [8]:
run("p4b_original_db_diff.py")


######## pipeline/sql/generation_rent.duckdb
country_year: db rows 588, csv rows 588, key match {'both': 588, 'left_only': 0, 'right_only': 0}
city_rent: db rows 552, csv rows 552, key match {'both': 552, 'left_only': 0, 'right_only': 0}
dim_country: db rows 52, csv rows 52, key match {'both': 52, 'left_only': 0, 'right_only': 0}
dim_year: db rows 31, csv rows 31, key match {'both': 31, 'left_only': 0, 'right_only': 0}
hicp_monthly: db rows 99856, csv rows 99856, key match {'both': 99856, 'left_only': 0, 'right_only': 0}
long_overburden: db rows 127035, csv rows 127035, key match {'both': 127035, 'left_only': 0, 'right_only': 0}
long_net_earn: db rows 199986, csv rows 199986, key match {'both': 199986, 'left_only': 0, 'right_only': 0}
long_with_parents: db rows 22086, csv rows 22086, key match {'both': 22086, 'left_only': 0, 'right_only': 0}

######## qa/sandbox/sql/generation_rent.duckdb
country_year: db rows 588, csv rows 588, key match {'both': 588, 'left_only': 0, 'right_only': 0}

## 5 — Business logic and reconciliation
`country_year`, `city_rent`, `dim_country`, `hicp_monthly` (wide) and `dim_year` rebuilt from the RAW files in DuckDB SQL (no pandas, no notebook code) and compared cell by cell; then 10 random records traced raw → final.

In [9]:
run("p5_independent_recompute.py")


### country_year_keys
  final_rows: 588
  recomputed_rows: 588
  keys_only_in_final: []
  keys_only_in_recomputed: []

### country_year_cells
  house_price_index_2015: {'exact': 588, 'mismatch': 0, 'null_mismatch': 0, 'max_abs_diff': 0.0, 'examples': []}
  house_price_index_2015_flag: {'exact': 588, 'mismatch': 0, 'null_mismatch': 0, 'max_abs_diff': 0.0, 'examples': []}
  net_earnings_eur: {'exact': 588, 'mismatch': 0, 'null_mismatch': 0, 'max_abs_diff': 0.0, 'examples': []}
  net_earnings_eur_flag: {'exact': 588, 'mismatch': 0, 'null_mismatch': 0, 'max_abs_diff': 0.0, 'examples': []}
  net_earnings_pps: {'exact': 588, 'mismatch': 0, 'null_mismatch': 0, 'max_abs_diff': 0.0, 'examples': []}
  net_earnings_pps_flag: {'exact': 588, 'mismatch': 0, 'null_mismatch': 0, 'max_abs_diff': 0.0, 'examples': []}
  price_to_income_vs_lt_avg: {'exact': 588, 'mismatch': 0, 'null_mismatch': 0, 'max_abs_diff': 0.0, 'examples': []}
  price_to_income_vs_lt_avg_flag: {'exact': 588, 'mismatch': 0, 'null_mi

In [10]:
run("p5_trace_records.py")


#1 house_price_index_2015 FI 2018 -> MATCH
   1 raw: eurostat-house-price-index/prc_hpi_a.csv line 4202: OBS_VALUE='103.34' OBS_FLAG=''
   2 long: long_hpi_a.csv line 4202: value=103.34 flag=-
   3 final: country_year.csv line 246: house_price_index_2015=103.34 flag=-

#2 net_earnings_eur LT 2005 -> MATCH
   1 raw: eurostat-net-earnings/earn_nt_net.csv line 29406: OBS_VALUE='3137.2' OBS_FLAG=''
   2 long: long_net_earn.csv line 29406: value=3137.2 flag=-
   3 final: country_year.csv line 359: net_earnings_eur=3137.2 flag=-

#3 net_earnings_pps LV 2012 -> MATCH
   1 raw: eurostat-net-earnings/earn_nt_net.csv line 160919: OBS_VALUE='7395.32' OBS_FLAG=''
   2 long: long_net_earn.csv line 160919: value=7395.32 flag=-
   3 final: country_year.csv line 408: net_earnings_pps=7395.32 flag=-

#4 price_to_income_vs_lt_avg RO 2011 -> MATCH
   1 raw: eurostat-house-price-to-income/tipsho60.csv line 1371: OBS_VALUE='100.59' OBS_FLAG=''
   2 long: long_ptir.csv line 1371: value=100.59 flag=-
   3 f

## 6 · Power BI readiness

In [11]:
run("p6_powerbi_readiness.py")


### country_year: 588 rows x 25 cols | PK ['geo', 'year'] dupes=0 nulls=0
   format: {'utf8_bom': True, 'valid_utf8': True, 'line_endings': 'CRLF', 'delimiter': ',', 'semicolons_in_file': 0, 'decimal_comma_values': 0, 'index_like_columns': [], 'bad_column_names': [], 'non_ascii_in_data': ''}
   geo                                {'inferred': 'text', 'blanks': 0, 'distinct': 28, 'mixed_numeric_and_text': False, 'leading_or_trailing_space': 0}
   country                            {'inferred': 'text', 'blanks': 0, 'distinct': 28, 'mixed_numeric_and_text': False, 'leading_or_trailing_space': 0}
   iso2                               {'inferred': 'text', 'blanks': 21, 'distinct': 28, 'mixed_numeric_and_text': False, 'leading_or_trailing_space': 0}
   year                               {'inferred': 'integer', 'blanks': 0, 'distinct': 21, 'mixed_numeric_and_text': False, 'leading_or_trailing_space': 0}
   house_price_index_2015             {'inferred': 'decimal', 'blanks': 59, 'distinct': 48

## 7 · Reproducibility

In [12]:
run("p7_reproducibility.py")

python: 3.13.17
platform: Windows-11-10.0.26200-SP0
libraries: {'pandas': '3.1.0', 'numpy': '2.5.4', 'duckdb': '2.0.0', 'matplotlib': '3.11.2', 'nbconvert': '7.17.2', 'nbformat': '5.11.1', 'ipykernel': '7.4.0', 'jupyter_client': '8.10.0'}
requirements_files_in_project: ['requirements.txt']
randomness_or_clock_in_pipeline: none found (no seeds needed)
fresh_rerun_outputs_identical: {'city_rent.csv': True, 'country_year.csv': True, 'dim_country.csv': True, 'dim_year.csv': True, 'hicp_monthly.csv': True, 'long_age_leave.csv': True, 'long_city_rents.csv': True, 'long_hicp.csv': True, 'long_hpi_a.csv': True, 'long_hpi_q.csv': True, 'long_net_earn.csv': True, 'long_overburden.csv': True, 'long_ptir.csv': True, 'long_with_parents.csv': True}
all_identical: True
run_in_original_layout: {'exit_code': 0, 'first_error_lines': []}
original_layout_outputs_identical: {'city_rent.csv': True, 'country_year.csv': True, 'dim_country.csv': True, 'dim_year.csv': True, 'hicp_monthly.csv': True, 'long_age_l

## 8 — Regression after the approved fixes (2026-11-04)
Old outputs (`qa/baseline_before_fixes/`) vs new outputs: proves the fixes changed only what they were meant to.

In [13]:
run("p8_fix_regression.py")


### long_tables_identical
  long_age_leave.csv: True
  long_city_rents.csv: True
  long_hicp.csv: True
  long_hpi_a.csv: True
  long_hpi_q.csv: True
  long_net_earn.csv: True
  long_overburden.csv: True
  long_ptir.csv: True
  long_with_parents.csv: True

### country_year
  rows_old: 588
  rows_new: 588
  keys_only_old: 0
  keys_only_new: 0
  changed_cells_by_column: {'iso2': 21}
  columns_added: []
  columns_removed: []
  only_iso2_changed: True
  iso2_changes_all_EU27_2020_to_blank: True

### city_rent
  rows_old: 552
  rows_new: 552
  keys_only_old: 0
  keys_only_new: 0
  changed_cells_by_column: {}
  columns_added: ['is_capital']
  columns_removed: []
  no_existing_cell_changed: True
  is_capital_equals_CAP_suffix: True

### dim_country
  rows_old: 52
  rows_new: 52
  keys_only_old: 0
  keys_only_new: 0
  changed_cells_by_column: {'is_aggregate': 1, 'iso2': 11}
  columns_added: ['in_country_year']
  columns_removed: []
  changes: {'is_aggregate': [('EEA', 'False', 'True')], 'iso2'

## Done
Findings, evidence, the prioritised fix list and the fix log / re-audit: **`qa/QA_REPORT.md`**.